# Comparación Final: Experimentos A vs B vs C

Este notebook presenta la **comparación integral** de los tres experimentos del proyecto NeuralPlumber:

| Exp | Descripción | Modelo |
|---|---|---|
| **A** | Baseline — DCGAN original (Volz et al.) | `netG_epoch_5000.pth` |
| **B** | Dataset expandido — DCGAN reentrenado con 2518 ventanas | `netG_15levels.pth` |
| **C** | CMA-ES + F3 — optimización en el espacio latente | `netG_15levels.pth` + CMA-ES |

**Pregunta central:** ¿Cuánto mejora cada estrategia la coherencia estructural de los niveles?

## 1. Setup — Imports y configuración de paths

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch

# ── Paths relativos desde notebooks/ ──────────────────────────────────────────
NOTEBOOK_DIR  = os.path.abspath('')
PROJECT_ROOT  = os.path.abspath('../../')
DAGSTUHL_PATH = os.path.join(PROJECT_ROOT, 'clone', 'DagstuhlGAN', 'pytorch')
SRC_PATH      = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'src')
BASELINE_DIR  = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'experiments', 'baseline')
EXPB_DIR      = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'experiments', 'expanded_data')
SF_DIR        = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'experiments', 'structural_fitness')
EXP_DIR       = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'experiments')

for p in [DAGSTUHL_PATH, SRC_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Python    :', sys.version)
print('PyTorch   :', torch.__version__)
print('pandas    :', pd.__version__)
print('DAGSTUHL  :', os.path.isdir(DAGSTUHL_PATH))
print('SRC_PATH  :', os.path.isdir(SRC_PATH))
print('BASELINE  :', os.path.isdir(BASELINE_DIR))
print('EXPB_DIR  :', os.path.isdir(EXPB_DIR))
print('SF_DIR    :', os.path.isdir(SF_DIR))

## 2. Tabla comparativa de métricas: Exp A vs B vs C

In [ ]:
# ── Cargar los 3 JSONs de métricas ────────────────────────────────────────────
with open(os.path.join(BASELINE_DIR, 'metrics_baseline.json'), 'r') as f:
    data_a = json.load(f)

with open(os.path.join(EXPB_DIR, 'metrics_baseline.json'), 'r') as f:
    data_b = json.load(f)

with open(os.path.join(SF_DIR, 'cmaes_results_netG_15levels_f3_static.json'), 'r') as f:
    data_c = json.load(f)

m_a = data_a['metrics']
m_b = data_b['metrics']
m_c = data_c['avg_metrics']
n_c = data_c['n_runs']

print(f'Exp A — n_samples : {data_a["n_samples"]}')
print(f'Exp B — n_samples : {data_b["n_samples"]}')
print(f'Exp C — n_runs    : {n_c}')

In [ ]:
# ── Construir tabla comparativa con pandas ────────────────────────────────────
METRIC_LABELS = {
    'pipe_completeness' : 'Completitud tuberías',
    'ground_continuity' : 'Continuidad suelo',
    'gap_traversability': 'Traversabilidad gaps',
    'enemy_placement'   : 'Colocación enemigos',
    'structural_avg'    : 'Promedio estructural',
}

rows = []
for key, label in METRIC_LABELS.items():
    va   = m_a[key]
    vb   = m_b[key]
    vc   = m_c[key]
    d_ac = vc - va
    pct  = (d_ac / va * 100) if va > 0 else float('nan')
    rows.append({
        'Métrica'       : label,
        'Exp A (base)' : round(va, 4),
        'Exp B (datos)' : round(vb, 4),
        'Exp C (CMA-ES)': round(vc, 4),
        'Δ A→C'         : round(d_ac, 4),
        '% mejora A→C'  : round(pct, 1),
    })

df = pd.DataFrame(rows).set_index('Métrica')

# Estilo para highlight de mejoras
def highlight_delta(val):
    if isinstance(val, float):
        if val >= 0.05:
            return 'background-color: #abebc6; font-weight: bold'
        elif val <= -0.05:
            return 'background-color: #f9c9c4'
    return ''

def highlight_pct(val):
    if isinstance(val, float):
        if val >= 20:
            return 'background-color: #abebc6; font-weight: bold'
        elif val <= -5:
            return 'background-color: #f9c9c4'
    return ''

display_df = df.style \
    .applymap(highlight_delta, subset=['Δ A→C']) \
    .applymap(highlight_pct,   subset=['% mejora A→C']) \
    .format({
        'Exp A (base)'  : '{:.4f}',
        'Exp B (datos)' : '{:.4f}',
        'Exp C (CMA-ES)': '{:.4f}',
        'Δ A→C'         : '{:+.4f}',
        '% mejora A→C'  : '{:+.1f}%',
    }) \
    .set_caption('Tabla comparativa — Métricas estructurales (n=100 para A y B; media de 40 runs para C)')

display(display_df)

print('\nResumen numérico (texto plano):')
print(df.to_string())

In [ ]:
# ── Resumen de mejoras clave ───────────────────────────────────────────────────
print('=' * 65)
print('RESUMEN: Mejoras de Exp A → Exp C (CMA-ES)')
print('=' * 65)

GOOD_THRESHOLD = 0.70

for key, label in METRIC_LABELS.items():
    va  = m_a[key]
    vc  = m_c[key]
    d   = vc - va
    pct = d / va * 100 if va > 0 else 0
    cross = ('  ** supera umbral 0.70 **'
              if va < GOOD_THRESHOLD and vc >= GOOD_THRESHOLD else '')
    print(f'  {label:<25}: {va:.4f} → {vc:.4f}  ({d:+.4f}, {pct:+.1f}%){cross}')

print('=' * 65)
print(f'\nExp B vs Exp A (structural_avg): {m_b["structural_avg"]:.4f} vs {m_a["structural_avg"]:.4f}'
      f'  (Δ = {m_b["structural_avg"]-m_a["structural_avg"]:+.4f})')
print(f'Exp C vs Exp A (structural_avg): {m_c["structural_avg"]:.4f} vs {m_a["structural_avg"]:.4f}'
      f'  (Δ = {m_c["structural_avg"]-m_a["structural_avg"]:+.4f})')

## 3. Gráfica comparativa de barras (comparison_metrics.png)

In [ ]:
# ── Mostrar imagen comparativa pre-generada ───────────────────────────────────
img_path = os.path.join(EXP_DIR, 'comparison_metrics.png')

if os.path.exists(img_path):
    img = mpimg.imread(img_path)
    h, w = img.shape[:2]
    fig_w = min(18, w / 72)
    fig_h = fig_w * h / w
    fig, ax = plt.subplots(figsize=(max(fig_w, 14), max(fig_h, 5)))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Comparación de métricas estructurales — Exp A vs B vs C',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f'Imagen cargada desde: {img_path}')
else:
    print(f'Imagen no encontrada: {img_path}')
    print('Generando gráfica inline...')

    metric_keys  = list(METRIC_LABELS.keys())
    metric_short = ['pipe\ncompleteness', 'ground\ncontinuity', 'gap\ntraversability',
                    'enemy\nplacement', 'structural\navg']
    vals_a = [m_a[k] for k in metric_keys]
    vals_b = [m_b[k] for k in metric_keys]
    vals_c = [m_c[k] for k in metric_keys]

    x = np.arange(len(metric_keys))
    w = 0.25

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - w,   vals_a, w, label='Exp A — Baseline',     color='#3498db', edgecolor='#222')
    ax.bar(x,       vals_b, w, label='Exp B — Dataset exp.', color='#e67e22', edgecolor='#222')
    ax.bar(x + w,   vals_c, w, label='Exp C — CMA-ES',       color='#e74c3c', edgecolor='#222')
    ax.axhline(0.70, color='#888', linestyle='--', linewidth=1.2, label='Umbral (0.70)')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_short, fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score (0 – 1)')
    ax.set_title('Comparación métricas estructurales — Exp A vs B vs C', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

## 4. Grid visual: 3 niveles representativos (uno por experimento)

In [ ]:
# ── Cargar nivel representativo de Exp A ──────────────────────────────────────
with open(os.path.join(BASELINE_DIR, 'sample_levels.json'), 'r') as f:
    sample_a_raw = json.load(f)

nivel_a = np.array(sample_a_raw[0], dtype=np.int32)  # primer nivel del baseline
print(f'Exp A — nivel representativo: shape={nivel_a.shape}')

In [ ]:
# ── Generar nivel de Exp B en vivo ────────────────────────────────────────────
# El modelo de Exp B es netG_15levels entrenado con dataset expandido
from models.dcgan import DCGAN_G, load_compatible

NZ, NC, NGF = 32, 10, 64

# Buscar el modelo de Exp B (netG_15levels_epoch_XXXX.pth)
models_dir  = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'models')
model_b_candidates = [
    os.path.join(DAGSTUHL_PATH, 'netG_15levels.pth'),
    os.path.join(models_dir,    'netG_15levels.pth'),
    os.path.join(EXPB_DIR,      'netG_15levels.pth'),
]
# También buscar con glob por nombre parcial
import glob as _glob
for g in [os.path.join(models_dir, 'netG_15levels*.pth'),
          os.path.join(DAGSTUHL_PATH, 'netG_15levels*.pth')]:
    model_b_candidates += _glob.glob(g)

model_b_path = next((p for p in model_b_candidates if os.path.exists(p)), None)

if model_b_path is not None:
    print(f'Modelo Exp B encontrado: {model_b_path}')
    netG_b = DCGAN_G(32, NZ, NC, NGF, ngpu=0, n_extra_layers=0)
    load_compatible(netG_b, model_b_path)
    netG_b.eval()
    torch.manual_seed(7)
    z_b = torch.randn(1, NZ, 1, 1)
    with torch.no_grad():
        out_b = netG_b(z_b)
    nivel_b = out_b[0, :, :14, :28].argmax(dim=0).numpy().astype(np.int32)
    print(f'Exp B — nivel generado: shape={nivel_b.shape}')
else:
    # Fallback: usar el modelo original de Exp A con otra semilla
    print('Modelo Exp B no encontrado. Usando modelo Exp A con semilla diferente (nivel B aproximado).')
    model_a_path = os.path.join(DAGSTUHL_PATH, 'netG_epoch_5000.pth')
    netG_b = DCGAN_G(32, NZ, NC, NGF, ngpu=0, n_extra_layers=0)
    load_compatible(netG_b, model_a_path)
    netG_b.eval()
    torch.manual_seed(123)
    z_b = torch.randn(1, NZ, 1, 1)
    with torch.no_grad():
        out_b = netG_b(z_b)
    nivel_b = out_b[0, :, :14, :28].argmax(dim=0).numpy().astype(np.int32)
    print(f'Exp B — nivel representativo (approx): shape={nivel_b.shape}')

In [ ]:
# ── Cargar mejor nivel del CMA-ES (Exp C) ────────────────────────────────────
with open(os.path.join(SF_DIR, 'best_levels_netG_15levels_f3_static.json'), 'r') as f:
    best_levels_raw = json.load(f)

nivel_c = np.array(best_levels_raw[0], dtype=np.int32)
print(f'Exp C — mejor nivel CMA-ES: shape={nivel_c.shape}')

In [ ]:
from visualization.level_renderer import render_comparison
from metrics.structural import structural_score

# ── Grid visual: 3 niveles en una figura ─────────────────────────────────────
levels_3 = [nivel_a, nivel_b, nivel_c]

ma = structural_score(nivel_a)
mb = structural_score(nivel_b)
mc = structural_score(nivel_c)

titles_3 = [
    f'Exp A — Baseline DCGAN\nstructural_avg={ma["structural_avg"]:.3f}',
    f'Exp B — Dataset expandido\nstructural_avg={mb["structural_avg"]:.3f}',
    f'Exp C — CMA-ES + F3\nstructural_avg={mc["structural_avg"]:.3f}',
]

render_comparison(levels_3, titles=titles_3)

In [ ]:
# ── Tabla de métricas de los 3 niveles representativos ────────────────────────
print('Métricas de los 3 niveles representativos:')
print('=' * 72)
print(f"{'Métrica':<33} {'Exp A':>8}  {'Exp B':>8}  {'Exp C':>8}")
print('=' * 72)

for key, label in METRIC_LABELS.items():
    va = ma[key]
    vb = mb[key]
    vc = mc[key]
    print(f'  {label:<31} {va:>8.4f}  {vb:>8.4f}  {vc:>8.4f}')

print('=' * 72)
print('(Nota: Exp A y B son niveles individuales; Exp C es el mejor del run 0)')

## 5. Convergencia del CMA-ES (cmaes_evolution.png)

In [ ]:
# ── Mostrar imagen de evolución del fitness ───────────────────────────────────
evo_path = os.path.join(SF_DIR, 'cmaes_evolution.png')

if os.path.exists(evo_path):
    img = mpimg.imread(evo_path)
    h, w = img.shape[:2]
    fig_w = min(18, w / 72)
    fig_h = fig_w * h / w
    fig, ax = plt.subplots(figsize=(max(fig_w, 14), max(fig_h, 5)))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Evolución del fitness CMA-ES — 40 runs independientes',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f'Imagen cargada desde: {evo_path}')
else:
    print(f'Imagen no encontrada: {evo_path}')
    print('Reconstruyendo curvas de evolución desde JSON...')

    with open(os.path.join(SF_DIR, 'cmaes_results_netG_15levels_f3_static.json'), 'r') as f:
        cmaes_data = json.load(f)

    fig, ax = plt.subplots(figsize=(14, 5))

    all_best_scores = []
    for run in cmaes_data['runs']:
        hist = run['fitness_history']
        gens = range(1, len(hist) + 1)
        ax.plot(gens, hist, alpha=0.3, linewidth=0.8, color='#3498db')
        all_best_scores.append(run['best_score'])

    # Mediana de curvas
    max_len = max(len(r['fitness_history']) for r in cmaes_data['runs'])
    padded  = []
    for run in cmaes_data['runs']:
        h = run['fitness_history']
        h_padded = h + [h[-1]] * (max_len - len(h))
        padded.append(h_padded)
    median_curve = np.median(padded, axis=0)
    ax.plot(range(1, max_len + 1), median_curve, color='#e74c3c',
            linewidth=2.5, label='Mediana (40 runs)', zorder=10)

    ax.set_xlabel('Generación CMA-ES')
    ax.set_ylabel('Fitness (f3_static, negativo)')
    ax.set_title('Evolución del fitness CMA-ES — 40 runs', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Análisis de convergencia numérico ─────────────────────────────────────────
with open(os.path.join(SF_DIR, 'cmaes_results_netG_15levels_f3_static.json'), 'r') as f:
    cmaes_data = json.load(f)

runs       = cmaes_data['runs']
best_scores = [r['best_score'] for r in runs]
n_evals_all = [r['n_evals']   for r in runs]
n_gens_all  = [len(r['fitness_history']) for r in runs]

print('Análisis de convergencia — 40 runs CMA-ES:')
print('=' * 50)
print(f'  Best score  — min  : {min(best_scores):.4f}')
print(f'  Best score  — media: {np.mean(best_scores):.4f}')
print(f'  Best score  — max  : {max(best_scores):.4f}')
print(f'  N evals     — media: {np.mean(n_evals_all):.1f}')
print(f'  N evals     — max  : {max(n_evals_all)}')
print(f'  N generac.  — media: {np.mean(n_gens_all):.1f}')
print(f'  N generac.  — max  : {max(n_gens_all)}')
print('=' * 50)

# Estimar generación de convergencia: cuando el fitness deja de mejorar >1%
conv_gens = []
for run in runs:
    hist = run['fitness_history']
    if len(hist) < 3:
        continue
    best_final = min(hist)
    for gi, val in enumerate(hist):
        if val <= best_final * 1.01:   # dentro del 1% del mejor
            conv_gens.append(gi + 1)
            break

if conv_gens:
    print(f'\n  Generación de convergencia (1% del mejor) — media: {np.mean(conv_gens):.1f}')
    print(f'  Generación de convergencia — mediana: {np.median(conv_gens):.0f}')
    print(f'  La mayoría de runs converge en ~{int(np.median(conv_gens))} generaciones')

### ¿Por qué converge CMA-ES tan rápido (~20 generaciones)?

1. **El espacio latente del DCGAN es suave y continuo.**  
   El generador fue entrenado para producir salidas suaves a partir de vectores latentes  
   gaussianos. Esto significa que vectores `z` cercanos producen niveles similares, lo que  
   crea un paisaje de fitness **localmente suave** y apto para optimización sin gradiente.

2. **CMA-ES es eficiente en espacios continuos de dimensión media (nz=32).**  
   Con solo 32 dimensiones, CMA-ES puede adaptar su covarianza en pocas generaciones  
   (~10·nz evaluaciones), localizando rápidamente las direcciones de mayor mejora.

3. **La función F3 tiene gradiente claro en las métricas que más varían**  
   (`pipe_completeness`, `ground_continuity`). Las regiones del espacio latente que  
   generan tuberías completas son suficientemente compactas para que CMA-ES las localice  
   en pocas iteraciones.

4. **El modelo ya tiene conocimiento estructural implícito.**  
   El DCGAN aprendió distribuciones de tiles correlacionados. CMA-ES no necesita  
   "inventar" estructuras nuevas, solo encontrar los vectores `z` que activan las  
   representaciones latentes correctas que el modelo ya conoce.

## 6. Conclusiones del Proyecto NeuralPlumber

---

### Hipótesis 1: Más datos de entrenamiento mejoran la coherencia estructural

**RESULTADO: NO CONFIRMADA**

El experimento B amplió el dataset de entrenamiento de **173 fragmentos** a **2518 ventanas**  
(factor 14.5×). Sin embargo, el promedio estructural prácticamente no cambió:

| | structural_avg |
|---|---|
| Exp A (173 fragmentos) | 0.6484 |
| Exp B (2518 ventanas)  | 0.6423 |
| **Diferencia**          | **−0.0061** |

**Interpretación:** La función de pérdida adversarial (WGAN-GP) optimiza la similitud de  
distribución estadística de los tiles, no las reglas estructurales de Mario. Con más datos  
el generador aprende una distribución más rica, pero sin métricas explícitas de coherencia  
en el entrenamiento, el resultado estructural no mejora significativamente.  
**La cantidad de datos no reemplaza a la función objetivo correcta.**

---

### Hipótesis 2: CMA-ES + F3 mejora la coherencia estructural

**RESULTADO: CONFIRMADA**

| Métrica | Exp A | Exp C | Δ | % mejora |
|---|---|---|---|---|
| `pipe_completeness`  | 0.2075 | **0.4567** | +0.2492 | **+120%** |
| `ground_continuity`  | 0.5093 | **0.8326** | +0.3233 | **+64%** |
| `gap_traversability` | 0.9100 | **1.0000** | +0.0900 | +10% |
| `enemy_placement`    | 0.9667 | **1.0000** | +0.0333 | +3% |
| **`structural_avg`** | **0.6484** | **0.8223** | **+0.1739** | **+27%** |

**Interpretación:**  
Optimizar directamente en el espacio latente con una función de fitness estructural explícita  
consigue resultados que el entrenamiento DCGAN por sí solo no puede alcanzar.  
La métrica `structural_avg` supera el umbral objetivo de **0.70** (alcanza **0.82**),  
y las métricas más difíciles (`pipe_completeness`, `ground_continuity`) muestran mejoras  
absolutas superiores al 60%.  
**CMA-ES explota la estructura suave del espacio latente del GAN sin necesidad de reentrenamiento.**

---

### Hallazgo adicional: el espacio latente del GAN tiene estructura suave

El CMA-ES converge en aproximadamente **20 generaciones** en la mayoría de runs,  
con un presupuesto de evaluaciones relativamente pequeño (~300–500 evaluaciones por run).  
Esto es evidencia directa de que:

- El espacio latente R³² del DCGAN tiene **gradiente suave** respecto a las métricas estructurales.
- Existen **regiones compactas** en el espacio latente que corresponden a niveles de alta calidad.
- La optimización sin gradiente es viable y eficiente para este tipo de generador.
- Los niveles "casi buenos" (DCGAN aleatorio) están **cerca en el espacio latente** de niveles  
  realmente buenos, lo que explica la rápida convergencia.

---

### Trabajo futuro

1. **F3 completo con agente Java (A\* simulator):**  
   La función F3 actual (`f3_static`) es una aproximación heurística que evalúa métricas  
   estáticas. La versión completa integraría un agente de Mario basado en A\* para evaluar  
   jugabilidad real (niveles completables, tiempo de completado, número de saltos requeridos).  
   Esto requiere la interfaz Java del Mario AI Framework y añadiría ~500ms por evaluación.

2. **Más runs y mayor presupuesto de evaluaciones:**  
   Con 40 runs y ~400 evaluaciones/run, el proyecto ya muestra resultados robustos.  
   Aumentar a 200 runs o 2000 evaluaciones/run podría revelar el techo real de calidad  
   alcanzable por CMA-ES en este espacio latente.

3. **Otros niveles y otros modelos:**  
   Extender el análisis a otros videojuegos del corpus VGLC (Mega Man, Kid Icarus,  
   Zelda) o usar arquitecturas más modernas (StyleGAN, WGAN-GP con condicionamiento)  
   como espacio de búsqueda para el CMA-ES.

4. **Optimización multi-objetivo:**  
   En lugar de agregar las métricas en un único escalar (F3), usar CMA-ES multi-objetivo  
   (MO-CMA-ES) para explorar el frente de Pareto entre `pipe_completeness` y diversidad.

5. **Análisis de diversidad:**  
   Los 40 niveles obtenidos por CMA-ES son estructuralmente buenos, pero ¿son diversos?  
   Combinar el fitness F3 con una métrica de novedad (novelty search) podría producir  
   colecciones de niveles con alta calidad **y** alta variedad.

In [ ]:
# ── Tabla final resumen del proyecto ──────────────────────────────────────────
import pandas as pd

summary_data = {
    'Experimento': ['Exp A — Baseline', 'Exp B — Dataset exp.', 'Exp C — CMA-ES + F3'],
    'Dataset'    : ['173 fragmentos', '2518 ventanas', '2518 ventanas'],
    'Estrategia' : ['DCGAN muestreo aleatorio', 'DCGAN reentrenado', 'CMA-ES en espacio latente'],
    'pipe_completeness': [m_a['pipe_completeness'], m_b['pipe_completeness'], m_c['pipe_completeness']],
    'ground_continuity': [m_a['ground_continuity'], m_b['ground_continuity'], m_c['ground_continuity']],
    'gap_traversability': [m_a['gap_traversability'], m_b['gap_traversability'], m_c['gap_traversability']],
    'enemy_placement': [m_a['enemy_placement'], m_b['enemy_placement'], m_c['enemy_placement']],
    'structural_avg': [m_a['structural_avg'], m_b['structural_avg'], m_c['structural_avg']],
    'Supera umbral 0.70': [
        m_a['structural_avg'] >= 0.70,
        m_b['structural_avg'] >= 0.70,
        m_c['structural_avg'] >= 0.70,
    ]
}

df_final = pd.DataFrame(summary_data).set_index('Experimento')

def style_avg(val):
    if isinstance(val, float) and val >= 0.70:
        return 'background-color: #abebc6; font-weight: bold'
    elif isinstance(val, float) and val < 0.50:
        return 'background-color: #f9c9c4'
    return ''

def style_bool(val):
    if val is True:
        return 'background-color: #abebc6; font-weight: bold'
    elif val is False:
        return 'background-color: #f9c9c4'
    return ''

display_final = df_final.style \
    .applymap(style_avg,  subset=['pipe_completeness', 'ground_continuity',
                                   'gap_traversability', 'enemy_placement', 'structural_avg']) \
    .applymap(style_bool, subset=['Supera umbral 0.70']) \
    .format({
        'pipe_completeness' : '{:.4f}',
        'ground_continuity' : '{:.4f}',
        'gap_traversability': '{:.4f}',
        'enemy_placement'   : '{:.4f}',
        'structural_avg'    : '{:.4f}',
    }) \
    .set_caption('Resumen final — NeuralPlumber: DCGAN + CMA-ES para generación de niveles de Mario')

display(display_final)

print('\n-- Conclusión final --')
print(f'Mejora de structural_avg: {m_a["structural_avg"]:.4f} → {m_c["structural_avg"]:.4f}')
print(f'Mejora relativa        : {(m_c["structural_avg"]-m_a["structural_avg"])/m_a["structural_avg"]*100:+.1f}%')
print(f'Umbral 0.70 superado por Exp C: {m_c["structural_avg"] >= 0.70}')